In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import json 
import pandas as pd
from collections import Counter
from scipy.stats import gaussian_kde
from copy import deepcopy
from constants import data_dir, meta_data_dir, blues, reds, output_dir, font_prop

In [ ]:
with open(meta_data_dir / Path("task_types.json"),"r") as f:
    task_types = json.load(f)

colors = {
    'pos': reds[3],
    'neg': blues[2],
}
edge_colors = {
    'pos': reds[-2],
    'neg': blues[-2],
}
legends = {
    "pos": "positive",
    "neg": "negtive",
}

# AA distribution

In [ ]:
os.makedirs(save_dir := (output_dir / Path("cls_aa")), exist_ok=True)

In [ ]:
for dataset_name, task_type in task_types.items():
    if task_type == "classification":
        data_info = pd.read_csv(data_dir / Path(dataset_name) / Path("enhanced_data.csv"))
        seqs = {
            "pos": [],
            "neg": []
        }
        [(seqs["pos"] if row["label"] else seqs["neg"]).append(row['sequence'].strip()) for _, row in data_info.iterrows()]
        aa_counts = {label: Counter("".join(seqs_per_label)) for label, seqs_per_label in seqs.items()}
        sorted_aa_types = list(sorted(set.union(*(set(aa_counts_per_label.keys()) for _, aa_counts_per_label in aa_counts.items()))))
        aa_type_to_index = {aa_type: index for index, aa_type in enumerate(sorted_aa_types)}
        sorted_counts = {
            label: [aa_counts_per_label.get(aa_type, 0) for aa_type in sorted_aa_types] for label, aa_counts_per_label in aa_counts.items()
        }
        x = np.arange(len(sorted_aa_types))
        plt.figure(figsize=(16, 6))
        for label, counts_per_label in sorted_counts.items():
            plt.bar(
                x, 
                counts_per_label, 
                color=colors[label], 
                width=1.0, 
                align='edge',
                edgecolor=edge_colors[label],
                alpha=0.5,
                label=legends[label],
            )

        plt.xticks(x+0.5, sorted_aa_types)
        plt.xlabel("Amino Acid Type", fontsize=14, fontproperties=font_prop)
        plt.ylabel("Occurrence", fontsize=14, fontproperties=font_prop)
        plt.xticks(fontsize=13, fontproperties=font_prop)
        plt.yticks(fontsize=13, fontproperties=font_prop)

        legend_font_prop = deepcopy(font_prop)
        legend_font_prop.set_size(14) 
        plt.legend(fontsize=20, prop=legend_font_prop)

        plt.savefig(
            save_dir / Path(f"{dataset_name}.png"),
            dpi=200, 
            bbox_inches="tight",
            transparent=True
        )